# Train a Model and Evaluate It

This notebook runs the benchmark scripts on a deliberately small MLP example, then inspects the artifacts they produce. The goal is to understand the train/evaluate workflow; full benchmark runs use the fixed configs under `configs/benchmark/best_models/` and usually run multiple seeds.

## Setup

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
import yaml

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

repo_root = Path("..").resolve()
gold_root = repo_root / "data" / "gold" / "lite"
tabular_dir = gold_root / "tabular"
core_dir = gold_root / "core"
run_dir = repo_root / "runs" / "tutorial_mlp" / "seed_00"


def read_yaml(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return yaml.safe_load(f) or {}


def display_command(command):
    shown = ["python" if idx == 0 else str(part) for idx, part in enumerate(command)]
    print(" ".join(shown))


gold_root

If Gold Lite is missing, download it from the repository root with:

```bash
python -m scripts.dataset.download.download_dataset --target gold_lite
```

## Benchmark Scripts

The production benchmark path is script-based. Training scripts fit a model, evaluation scripts create test predictions and metrics for one trained run, and test-evaluation scripts repeat train/eval over multiple seeds using fixed configs.

In [ ]:
pd.DataFrame(
    [
        {
            "role": "train one model",
            "example": "scripts/benchmark/train/mlp.py",
            "output": "model.pt and train_history.yaml",
        },
        {
            "role": "evaluate one trained run",
            "example": "scripts/benchmark/eval/mlp.py",
            "output": "test_predictions.parquet and eval_metrics.yaml",
        },
        {
            "role": "run repeated test evaluation",
            "example": "scripts/benchmark/test_eval/mlp.py",
            "output": "one run directory per seed plus test_eval_summary.yaml",
        },
        {
            "role": "fixed benchmark config",
            "example": "configs/benchmark/best_models/lite/mlp.yaml",
            "output": "hyperparameters used by the repeated test-eval script",
        },
    ]
)

Equivalent scripts live in the same folders for the other learning-based benchmark methods. For example, replace `mlp.py` with `xgboost.py`, `lstm.py`, `transformer.py`, or `gnn.py` when using the corresponding dataset and model family. Non-learning methods such as `translation.py` and `graph_event.py` do not have training scripts; they are run through the evaluation and test-evaluation scripts.

## Train a Small MLP

This run uses the real MLP training script, but with a tiny architecture and only three epochs so it is suitable for a tutorial. It is not the benchmark-quality configuration used for paper results.

In [ ]:
train_command = [
    sys.executable,
    "-m",
    "scripts.benchmark.train.mlp",
    "--data-dir",
    "data/gold/lite/tabular",
    "--output-dir",
    "runs/tutorial_mlp/seed_00",
    "--seed",
    "0",
    "--hidden-dims",
    "64,64",
    "--dropout",
    "0.0",
    "--epochs",
    "3",
    "--batch-size",
    "2048",
    "--num-workers",
    "0",
    "--val-fraction",
    "0.1",
    "--precision",
    "fp32",
    "--lr",
    "1e-3",
    "--weight-decay",
    "0.0",
    "--early-stopping-patience",
    "-1",
]

display_command(train_command)

In [ ]:
subprocess.run(train_command, cwd=repo_root, check=True)

`train_history.yaml` records the training loss and validation metrics from this run.

In [ ]:
train_history = read_yaml(run_dir / "train_history.yaml")
epochs = range(1, len(train_history["train_l1_per_epoch"]) + 1)

pd.DataFrame(
    [
        {
            "epoch": epoch,
            "train_l1": train_history["train_l1_per_epoch"][idx],
            "val_l1": train_history["val_l1_per_epoch"][idx],
            "val_mae_seconds": train_history["val_mae_seconds_per_epoch"][idx],
        }
        for idx, epoch in enumerate(epochs)
    ]
)

## Evaluate the Run

The evaluation script loads `model.pt`, predicts on the test split, converts predictions back to seconds, aligns them with the shared Gold core `test_eval_table.parquet`, and writes metrics.

In [ ]:
eval_command = [
    sys.executable,
    "-m",
    "scripts.benchmark.eval.mlp",
    "--data-dir",
    "data/gold/lite/tabular",
    "--test-eval-table",
    "data/gold/lite/core/test_eval_table.parquet",
    "--model-dir",
    "runs/tutorial_mlp/seed_00",
    "--batch-size",
    "2048",
]

display_command(eval_command)

In [ ]:
subprocess.run(eval_command, cwd=repo_root, check=True)

`eval_metrics.yaml` contains aggregate metrics and benchmark breakdowns. The table below shows the aggregate test metrics for the tutorial run.

In [ ]:
test_metrics = read_yaml(run_dir / "eval_metrics.yaml")["test"]

pd.DataFrame(
    [
        {"metric": "mae", "value": f"{test_metrics['mae']:.3f}"},
        {"metric": "rmse", "value": f"{test_metrics['rmse']:.3f}"},
        {"metric": "n_rows_matched", "value": f"{test_metrics['n_rows_matched']:,}"},
        {"metric": "n_rows_unmatched", "value": f"{test_metrics['n_rows_unmatched']:,}"},
        {"metric": "n_values_evaluated", "value": f"{test_metrics['n_values_evaluated']:,}"},
    ]
)

`test_predictions.parquet` contains the prediction table passed to the benchmark evaluator. Each row is keyed by snapshot timestamp, train id, and service date.

In [ ]:
pd.read_parquet(run_dir / "test_predictions.parquet").head()

## Repeated Test Evaluation

For benchmark-style runs, use the `test_eval` script. It resolves the fixed config for the selected tier, runs train+eval for each seed, and writes `test_eval_summary.yaml`.

In [ ]:
test_eval_command = [
    sys.executable,
    "-m",
    "scripts.benchmark.test_eval.mlp",
    "--data-dir",
    "data/gold/lite/tabular",
    "--test-eval-table",
    "data/gold/lite/core/test_eval_table.parquet",
    "--output-dir",
    "runs/mlp_lite_test_eval",
    "--tier",
    "lite",
    "--seeds",
    "0,1,2",
]

display_command(test_eval_command)

The command above is shown but not run automatically here, because it performs several complete train+eval runs. Use fewer seeds for a quick smoke test, or the full seed list for benchmark reporting.

[Back to README](../README.md#what-do-you-want-to-do)